In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# from IPython.display import display_latex

# Getting Started

We start by defining a `SystemDescriptor` object. The simplest way to create it is by means of the `build_system` function. By default, it creates a spin $1/2$ chain with four sites:



In [ ]:

from qalma import build_system, list_models_in_alps_xml, list_geometries_in_alps_xml, graph_from_alps_xml, model_from_alps_xml
system=build_system()


This creates a `SystemDescriptor` object, containing several properties. One of the is the `spec` property, storing information about the objects that defines it: 
* A model specification (`system.spec["model"]`)
* A geometry specification (`system.spec["graph"]`)
* A dictionary of parameters (`system.spec["parms"]`)

We can draw the lattice by accessing the `graph` element:


In [ ]:
system.spec["graph"].draw(plt)

We can also access the name of each site through the attribute `sites`. `system.sites` is a dictionary which associates to the name of each site its specification:


In [ ]:
print(system.sites.keys())

The site specification contains information like the dimension of the Hilbert space, the specification of the quantum numbera and the basic local operators defining the local algebra of observables:

In [ ]:
print("the dimension of the first site is ", system.sites['1[0]']["dimension"])
print("Quantum numbers:" ,system.sites['1[0]']["qn"])
print(tuple(system.sites['1[0]']["operators"]))

We can also access to operators defined over the whole system. Maybe the most important is the Hamiltonian:

In [ ]:
system.global_operator("Hamiltonian")

or the magnetization

In [ ]:
system.global_operator("Sz")

The list of predefined global operators can be accessed through 


In [ ]:
tuple(system.operators["global_operators"])

It is also possible to access to operators associated to a site:

In [ ]:
system.site_operator("Sx@1[0]")

Operators can be combined algebraically to build expressions. For example:

In [ ]:
H=system.global_operator("Hamiltonian")
sx1= system.site_operator("Sx@1[0]")
Hzeeman = -2 * system.global_operator("Sz") 
Htotal=(Hzeeman+H).simplify()
Htotal

Given an operator, it is straightforward to get its spectrum:

In [ ]:
Htotal.eigenenergies()

exponentiate it:

In [ ]:
Htotal.expm()

or get a trace:

In [ ]:
print("The partition function is ", (-Htotal).expm().tr(),"~", sum([np.exp(-en) for en in Htotal.eigenenergies()]))

To see over which sites acts an operator, we an use the function `qalma.utils.draw_operator(operator,ax)`:

In [ ]:
from qalma.utils import draw_operator
fig, ax = plt.subplots()
draw_operator(Htotal,ax)
Htotal.system.spec["graph"].draw(ax)

# Qutip integration


It is also straightforward to convert operators into qutip objects, and use them with the solvers:

In [ ]:
import qutip


sx01=system.site_operator("Sx@1[0]")+system.site_operator("Sx@1[1]")
rho0 = (sx01).expm()
rho0 = rho0/rho0.tr()
ts=np.linspace(0,10,100)
result = qutip.mesolve(tlist=ts, H=Hzeeman.to_qutip(), rho0=rho0.to_qutip(), e_ops=(sx01.to_qutip(),))
plt.plot(ts, result.e_data[0],label="$H_{Zeeman}$")
result = qutip.mesolve(tlist=ts, H=H.to_qutip(), rho0=rho0.to_qutip(), e_ops=(sx01.to_qutip(),))
plt.plot(ts, result.e_data[0],label="$H_{exc}$")
result = qutip.mesolve(tlist=ts, H=Htotal.to_qutip(), rho0=rho0.to_qutip(), e_ops=(sx01.to_qutip(),))
plt.plot(ts, result.e_data[0],label="$H_{total}$")
plt.legend()
plt.xlabel("t")
plt.ylabel(r"$\langle sx_1+sx_2\rangle$")


# Larger systems

As far as explicit computations requiring diagonalizations are not required, it is possible to define larger systems:

In [ ]:
system_large = build_system(a=1,L=100)
H=system_large.global_operator("Hamiltonian")
sz=system_large.global_operator("Sz")
sx0_loc=system_large.site_operator("Sx@1[0]")

(H*sx0_loc-sx0_loc*H).simplify()

In [ ]:
(H*sz-sz*H).simplify()

# Systems in other geometries

Other geometries models are also allowed

In [ ]:
    
from qalma.utils import eval_expr
from qalma.model import SystemDescriptor

# Load a system
system = build_system(geometry_name= "open square lattice",model_name="spin",  L=3, a=1, h=1,J=1) 


# enumerate the name of each subsystem
sites = [s for s in system.sites]

# Build some specific site and bond operators
exchange01=system.bond_operator("exchange_xy", sites[0], sites[1])
exchange10 =system.bond_operator("exchange_xy", sites[1], sites[0])
sz0 = system.site_operator("Sz",sites[0])
sz1 = system.site_operator("Sz",sites[1])
# Get the Hamiltonian 
H = system.global_operator("Hamiltonian")

# Plot the lattice and the spectrum
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 2, 1)

# Convert to qutip and get the spectrum
spectrum = H.to_qutip().eigenenergies()
ax.scatter(range(len(spectrum)),sorted(spectrum))
ax.set_title("spectrum")
ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.set_title("lattice")
system.spec["graph"].draw(ax, node_spec={"0":{"c":"lightgreen","s":100}})
system.spec["graph"].subgraph(frozenset({sites[0],sites[1],sites[3],})).draw(ax, node_spec={"0":{"c":"red","s":30}},
                                               edge_spec={"0":{"c":"orange"}})

In [ ]:
# Models and local basis

from qalma.settings import MODEL_LIB_FILE

models = list_models_in_alps_xml(MODEL_LIB_FILE)

for name in models:
    print(name)
    try:
        model = model_from_alps_xml(MODEL_LIB_FILE, name, parms={"L":3, "W":3, "a":1,"b":1, "c":1, "Nmax":5})
        print("site types:", {name: lb["name"]  for name, lb in  model.site_basis.items()})
    except Exception as e:
        print("   load failed")
    print(40*"-")

# Lattices in the library

In [ ]:
from qalma.settings import LATTICE_LIB_FILE
graphs = list_geometries_in_alps_xml(LATTICE_LIB_FILE)


fig = plt.figure(figsize=(20,40))
pos = 0
rows = (len(graphs)+2) //3
for name in graphs:
    pos += 1
    g = graph_from_alps_xml(LATTICE_LIB_FILE, name, parms={"L":3, "W":3, "a":1,"b":1, "c":1})
    
    if g.lattice and g.lattice["dimension"] > 2:
        ax = fig.add_subplot(rows, 3, pos, projection='3d')
        ax.set_proj_type("persp")
    else:
        ax = fig.add_subplot(rows, 3, pos)
    ax.set_title(name)
    g.draw(ax)
plt.show()

In [ ]:
# Systems
from qalma.model import SystemDescriptor


models = list_models_in_alps_xml(MODEL_LIB_FILE)
graphs = list_geometries_in_alps_xml(LATTICE_LIB_FILE)

for model_name in models:
    print(model_name, "\n", 10*"*")
    for graph_name in graphs:
        print([graph_name,model_name])
        g = graph_from_alps_xml(LATTICE_LIB_FILE, graph_name, parms={"L":3, "W":3, "a":1,"b":1, "c":1})
        model = model_from_alps_xml(MODEL_LIB_FILE, model_name, parms={"L":3, "W":3, "a":1,"b":1, "c":1, "Nmax":5})
        try:
            system = SystemDescriptor(g, model, {})
        except ValueError as e:
            print("   ", graph_name, "  [Failed]", type(e), e)
            continue
        print("   ", graph_name, "  [OK]")
    print("-------------")